In [1]:
from utility.plot_spectrum import plot_spectrum
from utility.adjust_redshift import adjust_redshift

from sparcl.client import SparclClient
from dl import queryClient as qc

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({"legend.frameon":False})

### prepare Hamann ERQ data

#### clean core ERQ dataframe

In [29]:
hamann_erq = pd.read_csv("data/Hamann2017-core-ERQ-stats.csv", skiprows=[1])


## rename columns
col_rename_dic = {
            "Quasar name":"SDSS_id",
            "N V/C IV":"NV/CIV",
            "αλ":"UV_slope",
            "E(B − V)":"E(B-V)",
            "i–W3":"i-W3"
            }

hamann_erq = hamann_erq.rename(columns=col_rename_dic)


## change weird - into nan
hamann_erq = hamann_erq.replace("–",np.nan)  


## split up uncertainties to new columns, turn into floats
hamann_erq["REW_unc"] = hamann_erq["REW"].apply(lambda x: x.split("±")[1]).astype("float")
hamann_erq["REW"] = hamann_erq["REW"].apply(lambda x: x.split("±")[0]).astype("float")

hamann_erq["FWHM_unc"] = hamann_erq["FWHM"].apply(lambda x: x.split("±")[1]).astype("float")
hamann_erq["FWHM"] = hamann_erq["FWHM"].apply(lambda x: x.split("±")[0]).astype("float")

hamann_erq["E(B-V)_unc"] = hamann_erq["E(B-V)"].astype(str).apply(lambda x: x.split("±"))  # has nan values hence the extra step inbetween
hamann_erq["E(B-V)"] = hamann_erq["E(B-V)_unc"].apply(lambda x: x[0]).astype("float")
hamann_erq["E(B-V)_unc"] = hamann_erq["E(B-V)_unc"].apply(lambda x: x[1] if len(x) > 1 else x[0]).astype("float")

hamann_erq["FIRST"] = hamann_erq["FIRST"].astype("float")


## turn that stubborn dumbass UV_slope column into floats, no idea why the sane way does not work
def convert_weird_minus_to_float(value):
    if value[0] == "−":
        value = value[1:]
        return float(value)*(-1)
    else:
        return float(value)

hamann_erq["UV_slope"] = hamann_erq["UV_slope"].apply(lambda x: convert_weird_minus_to_float(x))


## change column order
hamann_erq = hamann_erq[['SDSS_id', 'ze', 'i', 'i-W3', 'REW', 'REW_unc', 'FWHM', 'FWHM_unc', 'kt80', 'NV/CIV', 'BAL', 'UV_slope', 'E(B-V)', 'E(B-V)_unc', 'FIRST']]

hamann_erq

,SDSS_id,ze,i,i-W3,REW,REW_unc,FWHM,FWHM_unc,kt80,NV/CIV,BAL,UV_slope,E(B-V),E(B-V)_unc,FIRST
0,J000610.67+121501.2a,2.31,22.1,8.0,107.0,6.0,4540.0,200.0,0.37,1.88,0,-0.20,0.66,0.01,0.0
1,J000746.19+122223.9,2.43,21.3,4.9,220.0,6.0,2433.0,71.0,0.28,0.39,0,-0.17,0.19,0.01,0.0
2,J002400.25+245031.9,2.81,21.6,6.0,144.0,10.0,3523.0,195.0,0.37,2.46,0,-1.14,0.20,0.02,NaN
3,J004713.21+264024.7,2.56,20.4,4.7,101.0,2.0,3685.0,74.0,0.33,1.92,1,-1.08,0.15,0.01,NaN
4,J005044.95−021217.6,2.25,21.5,4.7,147.0,9.0,4343.0,487.0,0.19,0.61,0,-1.73,0.14,0.02,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,J223754.52+065026.6,2.61,22.0,5.8,141.0,5.0,1391.0,44.0,0.36,1.03,0,-1.20,0.24,0.02,0.0
93,J225438.30+232714.5,3.09,22.0,5.5,415.0,20.0,4412.0,146.0,0.36,0.66,0,-0.38,NaN,NaN,NaN
94,J232326.17−010033.1a,2.36,22.4,7.2,256.0,5.0,3989.0,62.0,0.37,2.16,0,0.91,0.40,0.01,0.0
95,J232828.47+044346.8,2.56,21.5,5.5,359.0,5.0,1584.0,20.0,0.35,0.37,0,-0.28,0.13,0.02,0.0


In [28]:
hamann_erq.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97 entries, 0 to 96
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   SDSS_id     97 non-null     object 
 1   ze          97 non-null     float64
 2   i           97 non-null     float64
 3   i-W3        97 non-null     float64
 4   REW         97 non-null     float64
 5   REW_unc     97 non-null     float64
 6   FWHM        97 non-null     float64
 7   FWHM_unc    97 non-null     float64
 8   kt80        97 non-null     float64
 9   NV/CIV      97 non-null     float64
 10  BAL         97 non-null     int64  
 11  UV_slope    97 non-null     float64
 12  E(B-V)      87 non-null     float64
 13  E(B-V)_unc  87 non-null     float64
 14  FIRST       91 non-null     float64
dtypes: float64(13), int64(1), object(1)
memory usage: 11.5+ KB


#### extract positions of ERQs from SDSS

### Find Hamann ERQ in DESI

In [ ]:
query = """
        SELECT targetid, target_ra, target_dec, z, 
        FROM desi_dr1.agnqso
        LIMIT 1000
        """

results = qc.query(sql=query, fmt="pandas")

In [16]:
results.head(50)

,targetid,ls_id
0,39633180159908726,9907736513026934
1,39633180159908110,9907736513026318
2,39633175927855592,9907736446900712
3,39633180159906587,9907736513024795
4,39633180159908251,9907736513026459
5,39633180164097697,9907736513087137
6,39633180164098202,9907736513087642
7,39633180159908219,9907736513026427
8,39633180164096180,9907736513085620
9,39633180164096335,9907736513085775


In [11]:
print(qc.schema('desi_dr1.zpix'))


Schema: desi_dr1
 Table: zpix

     Column Name   Description
     -----------   -----------
            elon   Ecliptic longitude
            elat   Ecliptic latitude
            glon   Galactic longitude
            glat   Galactic latitude
               z   Redshift measured by Redrock
            zerr   Redshift error from Redrock
            chi2   Best fit chi squared
         coeff_0   Redrock template coefficients
         coeff_1   Redrock template coefficients
         coeff_2   Redrock template coefficients
         coeff_3   Redrock template coefficients
         coeff_4   Redrock template coefficients
         coeff_5   Redrock template coefficients
         coeff_6   Redrock template coefficients
         coeff_7   Redrock template coefficients
         coeff_8   Redrock template coefficients
         coeff_9   Redrock template coefficients
       deltachi2   Delta-chi-squared for template fit from Redrock
   mean_fiber_ra   Mean (over exposures) RA of actual fiber posi